# Audit d’intégrité

**Objectif :** vérifier les identifiants et les contraintes
qui garantissent une structure fiable des données.

Cette étape consiste à :
- Proposer les clés primaires des fichiers clients et produits.
- Vérifier que les identifiants sont renseignés et uniques.
- Distinguer les lignes entièrement dupliquées des identifiants répétés.
- Définir des contraintes d’intégrité.
- Expliquer les conséquences des violations.
- Construire une table de suivi des violations.

Aucune correction n’est appliquée à cette étape.

In [1]:
import pandas as pd

clients_raw = pd.read_csv(
    "../data/raw/clients.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)

products_raw = pd.read_csv(
    "../data/raw/produits.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)

clients_df = clients_raw.copy()
products_df = products_raw.copy()

## 1. Proposition des clés primaires

| Fichier | Clé primaire proposée | Conditions attendues |
|---|---|---|
| Clients | `id_client` | Identifiant renseigné et unique pour chaque client. |
| Produits | `id_produit` | Identifiant renseigné et unique pour chaque produit. |

Le profilage a montré des répétitions de `CLI001` et de `PRD001`.
Ces champs ne peuvent donc pas être utilisés comme clés primaires
en l’état.

La prochaine étape consiste à vérifier leur non-nullité et à mesurer
les répétitions sur l’ensemble des données.

## 2. Vérification des identifiants

In [2]:
def check_identifiers(df, column):
    identifiers = df[column]

    missing = identifiers.isna() | identifiers.str.strip().eq("")
    repeated = identifiers.duplicated(keep=False) & ~missing

    print(f"Missing identifiers: {missing.sum()}")
    print(f"Repeated identifier values: {identifiers[repeated].nunique()}")
    print(f"Rows with repeated identifiers: {repeated.sum()}")

    display(df.loc[missing | repeated])

In [3]:
# Check client identifiers
check_identifiers(clients_df, "id_client")

Missing identifiers: 0
Repeated identifier values: 1
Rows with repeated identifiers: 2


,id_client,prenom,nom,age,email,telephone,ville,code_postal,type_client,date_inscription,entreprise_exemple
0,CLI001,Camille,Leroy,29,camille.leroy@example.com,06 12 34 50 01,Lille,59000,Particulier,2025-02-14,Les Saveurs du Nord (entreprise fictive)
1,CLI001,Thomas,Dubois,41,thomas.dubois@example.com,06 12 34 50 02,Roubaix,59100,Professionnel,2024-11-06,Les Saveurs du Nord (entreprise fictive)


In [4]:
# Check product identifiers
check_identifiers(products_df, "id_produit")

Missing identifiers: 0
Repeated identifier values: 1
Rows with repeated identifiers: 2


,id_produit,nom_produit,categorie,conditionnement,prix_unitaire_eur,tva_pct,stock_unites,statut_stock,type_fournisseur,entreprise_exemple
0,PRD001,Gaufres artisanales,Biscuits & douceurs,paquet 6 pièces,4.2,5.5,36.0,En stock,Fournisseur alimentaire,Les Saveurs du Nord (entreprise fictive)
1,PRD001,Spéculoos pur beurre,Biscuits & douceurs,sachet 250 g,3.6,5.5,70.0,En stock,Artisan partenaire,Les Saveurs du Nord (entreprise fictive)


## 3. Recherche des lignes entièrement dupliquées

In [5]:
def check_duplicate_rows(df):
    duplicated = df.duplicated(keep=False)

    print(f"Rows involved in exact duplicates: {duplicated.sum()}")
    print(f"Extra duplicate rows: {df.duplicated().sum()}")

    display(df.loc[duplicated])

In [6]:
# Check exact duplicate rows in the clients dataset
check_duplicate_rows(clients_df)

Rows involved in exact duplicates: 0
Extra duplicate rows: 0


,id_client,prenom,nom,age,email,telephone,ville,code_postal,type_client,date_inscription,entreprise_exemple


In [7]:
# Check exact duplicate rows in the products dataset
check_duplicate_rows(products_df)

Rows involved in exact duplicates: 0
Extra duplicate rows: 0


,id_produit,nom_produit,categorie,conditionnement,prix_unitaire_eur,tva_pct,stock_unites,statut_stock,type_fournisseur,entreprise_exemple


## 4. Contraintes d’intégrité proposées

| Contrainte | Champs concernés | Règle |
|---|---|---|
| Identifiants obligatoires | `id_client`, `id_produit` | Chaque identifiant doit être renseigné et ne pas contenir uniquement des espaces. |
| Identifiants uniques | `id_client`, `id_produit` | Un identifiant ne doit apparaître qu’une seule fois dans son fichier. |
| Prix non négatif | `prix_unitaire_eur` | Chaque prix renseigné doit être numérique et supérieur ou égal à zéro. Un prix nul est accepté si les produits gratuits sont autorisés. |
| Stock entier et non négatif | `stock_unites` | Chaque quantité renseignée doit être un entier supérieur ou égal à zéro, si les stocks négatifs ne sont pas autorisés. |
| TVA autorisée | `tva_pct` | Chaque taux renseigné doit appartenir à la liste des taux autorisés pour les produits concernés. |

Les contrôles montrent qu’aucun identifiant n’est manquant, mais que
l’unicité n’est pas respectée dans les deux fichiers.

Le profilage a repéré un prix de -5,20 €, un stock de -12 unités
et un taux de TVA de 99 %. Ces valeurs sont à examiner au regard
des règles proposées, qui restent à confirmer avec le métier.

Aucune contrainte de clé étrangère n’est proposée entre les deux fichiers :
aucun champ fourni ne relie directement un client à un produit.

## 5. Conséquences possibles d’une violation d’intégrité

| Conséquence | Explication | Exemple |
|---|---|---|
| Fusion de deux entités | Deux clients ou produits différents peuvent être considérés comme une seule entité. | `CLI001` désigne Camille Leroy et Thomas Dubois : leurs informations risquent d’être mélangées. |
| Comptage erroné | Les répétitions peuvent fausser les totaux ou le nombre d’entités distinctes. | Compter les identifiants clients distincts sous-estime le nombre de clients si deux personnes partagent le même identifiant. |
| Perte de traçabilité | Il devient difficile de savoir à quelle entité appartient une opération. | Une modification associée à `PRD001` ne permet pas de déterminer avec certitude quel produit est concerné. |
| Difficulté de jointure | Une jointure sur un identifiant répété peut produire plusieurs correspondances et multiplier les lignes. | Une ligne de commande portant `PRD001` pourrait être associée aux deux produits qui partagent cet identifiant. |

## 6. Table de suivi des violations

In [8]:
def track_integrity_checks(df, file_name, id_column, checks):
    results = []

    for field, rule, mask, status in checks:
        for position, flagged in enumerate(mask, start=1):
            if flagged:
                results.append({
                    "file": file_name,
                    "row_number": position,
                    "identifier": df[id_column].iloc[position - 1],
                    "field": field,
                    "rule": rule,
                    "test_result": status,
                })

    return pd.DataFrame(
        results,
        columns=[
            "file", "row_number", "identifier",
            "field", "rule", "test_result",
        ],
    )

In [9]:
def identifier_checks(df, id_column):
    identifiers = df[id_column]
    missing = identifiers.isna() | identifiers.str.strip().eq("")

    return [
        (
            id_column,
            "Identifier must be populated",
            missing,
            "FAIL",
        ),
        (
            id_column,
            "Identifier must be unique",
            identifiers.duplicated(keep=False) & ~missing,
            "FAIL",
        ),
        (
            "all_columns",
            "Full row must not be duplicated",
            df.duplicated(keep=False),
            "FAIL",
        ),
    ]

In [10]:
prices = pd.to_numeric(products_df["prix_unitaire_eur"], errors="coerce")
stock = pd.to_numeric(products_df["stock_unites"], errors="coerce")

product_checks = identifier_checks(products_df, "id_produit")

product_checks += [
    (
        "prix_unitaire_eur",
        "Populated price must be numeric and non-negative",
        products_df["prix_unitaire_eur"].str.strip().ne("")
        & (prices.isna() | prices.lt(0)),
        "REVIEW",
    ),
    (
        "stock_unites",
        "Populated stock must be a non-negative whole number",
        products_df["stock_unites"].str.strip().ne("")
        & (stock.isna() | stock.lt(0) | stock.mod(1).ne(0)),
        "REVIEW",
    ),
]

In [11]:
integrity_violations = pd.concat(
    [
        track_integrity_checks(
            clients_df,
            "clients.csv",
            "id_client",
            identifier_checks(clients_df, "id_client"),
        ),
        track_integrity_checks(
            products_df,
            "produits.csv",
            "id_produit",
            product_checks,
        ),
    ],
    ignore_index=True,
)

display(integrity_violations)

,file,row_number,identifier,field,rule,test_result
0,clients.csv,1,CLI001,id_client,Identifier must be unique,FAIL
1,clients.csv,2,CLI001,id_client,Identifier must be unique,FAIL
2,produits.csv,1,PRD001,id_produit,Identifier must be unique,FAIL
3,produits.csv,2,PRD001,id_produit,Identifier must be unique,FAIL
4,produits.csv,10,PRD010,prix_unitaire_eur,Populated price must be numeric and non-negative,REVIEW
5,produits.csv,40,PRD040,stock_unites,Populated stock must be a non-negative whole n...,REVIEW


**Lecture des résultats :**
- `FAIL` : une contrainte structurelle confirmée n’est pas respectée.
- `REVIEW` : une valeur ne respecte pas une contrainte proposée,
  qui reste à confirmer avec le métier.

Le numéro de ligne commence à 1, sans compter l’en-tête du CSV.

Le contrôle de TVA n’est pas encore exécuté : la liste des taux autorisés
doit d’abord être définie. Le taux de 99 % reste un point à vérifier.

Les contrôles sans violation restent documentés dans les cellules
précédentes, notamment l’absence d’identifiants manquants et de lignes
entièrement dupliquées.

## Synthèse de l’audit d’intégrité

- Aucun identifiant manquant et aucune ligne entièrement dupliquée.
- Un identifiant répété dans chaque fichier, concernant deux lignes clients et deux lignes produits : **4 violations d’unicité**.
- Un prix négatif et un stock négatif : **2 points à examiner** selon les contraintes métier proposées.
- Le contrôle des taux de TVA reste à définir : aucune liste de taux autorisés n’a été validée.

Les 6 lignes de suivi correspondent à 4 résultats `FAIL` et 2 résultats `REVIEW`, et non à 6 violations confirmées.